# 🎮 Ultimate Tic-Tac-Toe — IA Minimax + Alpha-Beta

**Projet Battle IA — Fondements de l'IA**  
Algorithme : Minimax avec élagage Alpha-Beta  
Format des coups : **colonne puis ligne** (1-9)

---
### Modes disponibles dans ce notebook
| Cellule | Mode |
|---|---|
| Section 4 | 🤖 IA vs IA (test interne) |
| Section 5 | ⚔️ **Battle inter-Colab** (combat officiel) |
| Section 6 | 🧑 Humain vs IA |

> **Pour le combat officiel**, utilisez la **Section 5** uniquement.

## 1. 🔧 Installation et code source
Exécutez cette cellule **en premier**. Elle définit tout le code du projet en mémoire.

In [1]:
# ============================================================
# BOARD — Représentation de l'état du jeu
# ============================================================

EMPTY    = 0
PLAYER_X = 1   # croix  (commence en premier)
PLAYER_O = 2   # ronds
DRAW     = 3   # match nul local


class Board:
    """
    État complet d'une partie d'Ultimate Tic Tac Toe.
    cells[gr][gc][lr][lc]  gr,gc = sous-grille (0..2)  lr,lc = case (0..2)
    """
    def __init__(self):
        self.cells        = [[[[EMPTY]*3 for _ in range(3)] for _ in range(3)] for _ in range(3)]
        self.local_winner = [[EMPTY]*3 for _ in range(3)]
        self.active_grid  = None          # None = libre
        self.current_player = PLAYER_X
        self.global_winner  = EMPTY
        self.move_count     = 0

    def copy(self):
        new = Board.__new__(Board)
        new.cells = [[
            [[self.cells[gr][gc][lr][lc] for lc in range(3)] for lr in range(3)]
            for gc in range(3)] for gr in range(3)]
        new.local_winner    = [row[:] for row in self.local_winner]
        new.active_grid     = self.active_grid
        new.current_player  = self.current_player
        new.global_winner   = self.global_winner
        new.move_count      = self.move_count
        return new

    def opponent(self, player=None):
        p = player if player is not None else self.current_player
        return PLAYER_O if p == PLAYER_X else PLAYER_X

    def is_terminal(self):         return self.global_winner != EMPTY
    def is_subgrid_done(self,g,c): return self.local_winner[g][c] != EMPTY

print('[OK] Board chargé')

[OK] Board chargé


In [2]:
# ============================================================
# RULES — Moteur de règles (Actions, Result, Terminal-test)
# ============================================================

WINNING_LINES = [
    [(0,0),(0,1),(0,2)], [(1,0),(1,1),(1,2)], [(2,0),(2,1),(2,2)],
    [(0,0),(1,0),(2,0)], [(0,1),(1,1),(2,1)], [(0,2),(1,2),(2,2)],
    [(0,0),(1,1),(2,2)], [(0,2),(1,1),(2,0)],
]

def _check_winner_3x3(grid):
    for line in WINNING_LINES:
        vals = [grid[r][c] for r,c in line]
        if vals[0] != EMPTY and vals[0] == vals[1] == vals[2]:
            return vals[0]
    return EMPTY

def _is_full_3x3(grid):
    return all(grid[r][c] != EMPTY for r in range(3) for c in range(3))

def _update_local_winner(board, gr, gc):
    if board.local_winner[gr][gc] != EMPTY: return
    w = _check_winner_3x3(board.cells[gr][gc])
    if w != EMPTY:                       board.local_winner[gr][gc] = w
    elif _is_full_3x3(board.cells[gr][gc]): board.local_winner[gr][gc] = DRAW

def _update_global_winner(board):
    if board.global_winner != EMPTY: return
    meta = [[board.local_winner[gr][gc] if board.local_winner[gr][gc]!=DRAW else EMPTY
             for gc in range(3)] for gr in range(3)]
    w = _check_winner_3x3(meta)
    if w != EMPTY: board.global_winner = w; return
    if all(board.local_winner[gr][gc]!=EMPTY for gr in range(3) for gc in range(3)):
        board.global_winner = DRAW

def get_legal_moves(board):
    if board.is_terminal(): return []
    ag = board.active_grid
    if ag is not None and not board.is_subgrid_done(ag[0], ag[1]):
        grids = [ag]
    else:
        grids = [(gr,gc) for gr in range(3) for gc in range(3)
                 if not board.is_subgrid_done(gr,gc)]
    return [(gr,gc,lr,lc) for gr,gc in grids
            for lr in range(3) for lc in range(3)
            if board.cells[gr][gc][lr][lc] == EMPTY]

def apply_move(board, move):
    gr,gc,lr,lc = move
    b = board.copy()
    b.cells[gr][gc][lr][lc] = board.current_player
    b.move_count += 1
    _update_local_winner(b, gr, gc)
    _update_global_winner(b)
    if not b.is_terminal():
        b.active_grid = (lr,lc) if not b.is_subgrid_done(lr,lc) else None
    else:
        b.active_grid = None
    b.current_player = board.opponent()
    return b

print('[OK] Rules chargé')

[OK] Rules chargé


In [3]:
# ============================================================
# DISPLAY — Affichage texte 9x9
# ============================================================

_SYM     = {EMPTY:'.', PLAYER_X:'X', PLAYER_O:'O'}
_WIN_SYM = {PLAYER_X:'X', PLAYER_O:'O', DRAW:'='}

def display_board(board, last_move=None):
    print()
    print('    1  2  3     4  5  6     7  8  9')
    print()
    for gr in range(3):
        if gr > 0:
            print('   ' + '===========' * 3)
        for lr in range(3):
            row = gr*3+lr+1
            s = f'{row}  '
            for gc in range(3):
                if gc > 0: s += ' || '
                if board.local_winner[gr][gc] != EMPTY:
                    sym = _WIN_SYM[board.local_winner[gr][gc]]
                    s += f'  {sym}  {sym}  ' if lr==1 else '         '
                else:
                    for lc in range(3):
                        if lc > 0: s += '|'
                        cell = board.cells[gr][gc][lr][lc]
                        sym  = _SYM[cell]
                        s += f'[{sym}]' if last_move and (gr,gc,lr,lc)==last_move else f' {sym} '
            print(s)
    print()
    print('    1  2  3     4  5  6     7  8  9')
    # Statut
    if board.global_winner == EMPTY:
        ag = board.active_grid
        zone = f'sous-grille ({ag[0]+1},{ag[1]+1})' if ag else 'libre'
        print(f"  ▶ Tour de {'X' if board.current_player==PLAYER_X else 'O'} | Zone : {zone} | Coups : {board.move_count}")
    elif board.global_winner == DRAW:
        print(f'  ═══ MATCH NUL ({board.move_count} coups) ═══')
    else:
        print(f"  ★ VICTOIRE de {'X' if board.global_winner==PLAYER_X else 'O'} ({board.move_count} coups) ★")
    print()

def parse_move(col_str, row_str, board):
    """Parse 'colonne ligne' (1-9) -> (gr,gc,lr,lc) ou None."""
    try:
        col, row = int(col_str)-1, int(row_str)-1
    except ValueError:
        return None
    if not (0<=col<=8 and 0<=row<=8): return None
    move = (row//3, col//3, row%3, col%3)
    return move if move in get_legal_moves(board) else None

print('[OK] Display chargé')

[OK] Display chargé


In [4]:
# ============================================================
# HEURISTIQUE — Fonction d'évaluation
# ============================================================

SCORE_WIN   = 10000
SCORE_LOCAL = 100
SCORE_DRAW_LOCAL = -10
SCORE_LINE  = 10

POS_GLOBAL = [[20,10,20],[10,30,10],[20,10,20]]
POS_LOCAL  = [[ 2, 1, 2],[ 1, 3, 1],[ 2, 1, 2]]

def _line_score(grid, player, opp):
    score = 0
    for line in WINNING_LINES:
        vals = [grid[r][c] for r,c in line]
        pc, oc = vals.count(player), vals.count(opp)
        if oc == 0:
            if pc == 2: score += SCORE_LINE
            elif pc == 1: score += 1
        if pc == 0:
            if oc == 2: score -= SCORE_LINE
            elif oc == 1: score -= 1
    return score

def _eval_subgrid(grid, player, opp):
    score = _line_score(grid, player, opp)
    for r in range(3):
        for c in range(3):
            if grid[r][c] == player:  score += POS_LOCAL[r][c]
            elif grid[r][c] == opp:   score -= POS_LOCAL[r][c]
    return score

def evaluate(board, player=PLAYER_X):
    opp = board.opponent(player)
    if board.global_winner == player: return  SCORE_WIN
    if board.global_winner == opp:    return -SCORE_WIN
    if board.global_winner == DRAW:   return  0
    score = 0
    meta = [[board.local_winner[gr][gc] if board.local_winner[gr][gc]!=DRAW else EMPTY
             for gc in range(3)] for gr in range(3)]
    for gr in range(3):
        for gc in range(3):
            lw = board.local_winner[gr][gc]
            pos = POS_GLOBAL[gr][gc]
            if   lw == player: score += SCORE_LOCAL + pos
            elif lw == opp:    score -= SCORE_LOCAL + pos
            elif lw == DRAW:   score += SCORE_DRAW_LOCAL
            else:              score += _eval_subgrid(board.cells[gr][gc], player, opp)
    score += _line_score(meta, player, opp) * 5
    return score

def terminal_score(board, player):
    opp = board.opponent(player)
    if board.global_winner == player: return  SCORE_WIN
    if board.global_winner == opp:    return -SCORE_WIN
    return 0

print('[OK] Heuristique chargée')

[OK] Heuristique chargée


In [5]:
# ============================================================
# MINIMAX + ALPHA-BETA — Décision de l'IA
# ============================================================

import math, time

_PRIORITY = {(1,1):0,(0,0):1,(0,2):2,(2,0):3,(2,2):4,
             (0,1):5,(1,0):6,(1,2):7,(2,1):8}

def _order(move):
    return _PRIORITY.get((move[2], move[3]), 9)

def minimax(board, depth, alpha, beta, maximizing, root_player):
    if board.is_terminal(): return terminal_score(board, root_player)
    if depth == 0:          return evaluate(board, root_player)
    moves = sorted(get_legal_moves(board), key=_order)
    if not moves:           return evaluate(board, root_player)

    if maximizing:
        v = -math.inf
        for m in moves:
            v = max(v, minimax(apply_move(board,m), depth-1, alpha, beta, False, root_player))
            alpha = max(alpha, v)
            if alpha >= beta: break
        return v
    else:
        v = math.inf
        for m in moves:
            v = min(v, minimax(apply_move(board,m), depth-1, alpha, beta, True, root_player))
            beta = min(beta, v)
            if alpha >= beta: break
        return v

def get_best_move(board, depth=4):
    player = board.current_player
    moves  = sorted(get_legal_moves(board), key=_order)
    if not moves:    return None
    if len(moves)==1: return moves[0]
    best, best_v = moves[0], -math.inf
    alpha, beta  = -math.inf, math.inf
    for m in moves:
        v = minimax(apply_move(board,m), depth-1, alpha, beta, False, player)
        if v > best_v: best_v, best = v, m
        alpha = max(alpha, best_v)
        if best_v >= SCORE_WIN: break
    return best

def get_best_move_timed(board, time_limit=5.0, max_depth=8):
    """Approfondissement itératif : maximise la profondeur dans le temps imparti."""
    moves = get_legal_moves(board)
    if not moves:    return None
    if len(moves)==1: return moves[0]
    best  = sorted(moves, key=_order)[0]
    start = time.time()
    for depth in range(1, max_depth+1):
        if time.time()-start >= time_limit: break
        m = get_best_move(board, depth=depth)
        if m: best = m
        if apply_move(board, best).is_terminal(): break
    elapsed = time.time()-start
    print(f'  [IA] profondeur atteinte={depth} | temps={elapsed:.2f}s')
    return best

print('[OK] Minimax + Alpha-Beta chargé')

[OK] Minimax + Alpha-Beta chargé


## 2. ⚙️ Configuration de l'IA
Modifiez ici les paramètres de votre IA avant le combat.

In [6]:
# ============================================================
# CONFIGURATION — Adaptez ces paramètres
# ============================================================

NOM_IA         = 'MonIA'      # nom de votre IA (affiché pendant le combat)
TIME_LIMIT     = 5.0          # secondes max par coup (recommandé: 2-6s, max 10s)
MAX_DEPTH      = 8            # profondeur max pour l'approfondissement itératif
DEPTH_FIXE     = 4            # profondeur fixe (si USE_TIMED=False)
USE_TIMED      = True         # True = limite temps | False = profondeur fixe

def mon_ia_joue(board):
    """Fonction principale : retourne le meilleur coup pour l'état `board`."""
    if USE_TIMED:
        return get_best_move_timed(board, time_limit=TIME_LIMIT, max_depth=MAX_DEPTH)
    else:
        return get_best_move(board, depth=DEPTH_FIXE)

print(f'[OK] IA configurée : {NOM_IA}')
print(f'     Mode : {"Timed (" + str(TIME_LIMIT) + "s)" if USE_TIMED else "Profondeur fixe=" + str(DEPTH_FIXE)}')

[OK] IA configurée : MonIA
     Mode : Timed (5.0s)


## 3. 🧪 Tests rapides
Vérification que tout fonctionne avant le combat.

In [7]:
# ============================================================
# TESTS RAPIDES
# ============================================================
errors = 0

# Test 1 : coups légaux au départ
b = Board()
assert len(get_legal_moves(b)) == 81, 'ERREUR: coups légaux initiaux'
print('[OK] 81 coups légaux au départ')

# Test 2 : apply_move immuable
b2 = apply_move(b, (1,1,1,1))
assert b.cells[1][1][1][1] == EMPTY, 'ERREUR: apply_move modifie le board original'
assert b2.cells[1][1][1][1] == PLAYER_X
assert b2.active_grid == (1,1)
print('[OK] apply_move immuable, active_grid correct')

# Test 3 : victoire locale
b3 = Board()
b3.cells[0][0][0][0] = b3.cells[0][0][1][0] = b3.cells[0][0][2][0] = PLAYER_X
_update_local_winner(b3, 0, 0)
assert b3.local_winner[0][0] == PLAYER_X, 'ERREUR: victoire locale'
print('[OK] Victoire locale détectée')

# Test 4 : IA joue un coup légal
b4 = Board()
import time
t0 = time.time()
move = mon_ia_joue(b4)
dt   = time.time()-t0
assert move in get_legal_moves(b4), 'ERREUR: coup IA illégal'
print(f'[OK] IA joue coup légal {move} en {dt:.2f}s')

# Test 5 : parse_move
b5 = Board()
m = parse_move('5','5',b5)   # centre absolu = (1,1,1,1)
assert m == (1,1,1,1), f'ERREUR parse_move: {m}'
print('[OK] parse_move(5,5) = (1,1,1,1) centre global')

print('\n✅ Tous les tests passent — prêt pour le combat !')

[OK] 81 coups légaux au départ
[OK] apply_move immuable, active_grid correct
[OK] Victoire locale détectée
  [IA] profondeur atteinte=8 | temps=8.58s
[OK] IA joue coup légal (1, 1, 1, 1) en 8.58s
[OK] parse_move(5,5) = (1,1,1,1) centre global

✅ Tous les tests passent — prêt pour le combat !


## 4. ⚔️ Mode Battle inter-Colab (combat officiel)

Ce mode est utilisé pendant les combats officiels.  
**Principe** : un humain relaie les coups entre les deux Colabs.

### Déroulement
1. Déterminer qui est **X** (commence) et qui est **O**
2. Configurer `MON_ROLE` ci-dessous
3. Exécuter la cellule de combat
4. Quand c'est votre tour : l'IA calcule et affiche son coup → vous le dictez à l'adversaire
5. Quand c'est le tour adverse : saisissez le coup reçu

In [8]:
# ============================================================
# CONFIGURATION DU COMBAT OFFICIEL
# ============================================================

MON_ROLE = PLAYER_X   # PLAYER_X si vous commencez, PLAYER_O sinon
                       # Changez en PLAYER_O si vous êtes le second

NOM_ADVERSAIRE = 'Adversaire'

print(f'Configuration : {NOM_IA} joue {"X" if MON_ROLE==PLAYER_X else "O"}')
print(f'Adversaire    : {NOM_ADVERSAIRE}')

Configuration : MonIA joue X
Adversaire    : Adversaire


In [ ]:
# ============================================================
# COMBAT OFFICIEL — Boucle de jeu inter-Colab
# ============================================================
# Relancez cette cellule pour chaque nouvelle partie.

board     = Board()
last_move = None
history   = []
timings   = []

print('=' * 55)
print(f'  ⚔️  {NOM_IA} ({"X" if MON_ROLE==PLAYER_X else "O"}) vs {NOM_ADVERSAIRE}')
print('=' * 55)
display_board(board)

while not board.is_terminal():

    if board.current_player == MON_ROLE:
        # ---- MON TOUR : l'IA calcule ----
        print(f'\n🤖 {NOM_IA} réfléchit...')
        t0   = time.time()
        move = mon_ia_joue(board)
        dt   = time.time() - t0
        timings.append(dt)

        gr,gc,lr,lc = move
        col_out = gc*3 + lc + 1
        row_out = gr*3 + lr + 1

        print(f'\n┌─────────────────────────────────────┐')
        print(f'│  MON COUP  →  colonne {col_out}, ligne {row_out}         │')
        print(f'│  (temps : {dt:.2f}s)                      │')
        print(f'└─────────────────────────────────────┘')
        print(f'  👉 Dictez ce coup à votre adversaire.')

    else:
        # ---- TOUR ADVERSE : saisie manuelle ----
        ag = board.active_grid
        if ag:
            print(f'\n⌨️  Coup adversaire (zone : sous-grille ({ag[0]+1},{ag[1]+1})) :')
        else:
            print(f'\n⌨️  Coup adversaire (libre) :')

        move = None
        while move is None:
            raw = input('   colonne (1-9) : ').strip()
            lig = input('   ligne   (1-9) : ').strip()
            move = parse_move(raw, lig, board)
            if move is None:
                print('   ⚠️  Coup invalide — réessayez.')

    board     = apply_move(board, move)
    last_move = move
    history.append(move)
    display_board(board, last_move=last_move)

# ---- Résultat ----
print('\n' + '=' * 55)
w = board.global_winner
if   w == MON_ROLE:           print(f'  🏆 VICTOIRE de {NOM_IA} !')
elif w == DRAW:                print('  🤝 MATCH NUL')
else:                          print(f'  💀 Défaite de {NOM_IA}')
print(f'  Coups joués : {board.move_count}')
if timings:
    print(f'  Temps moyen IA : {sum(timings)/len(timings):.2f}s | max : {max(timings):.2f}s')
print('=' * 55)

  ⚔️  MonIA (X) vs Adversaire

    1  2  3     4  5  6     7  8  9

1   . | . | .  ||  . | . | .  ||  . | . | . 
2   . | . | .  ||  . | . | .  ||  . | . | . 
3   . | . | .  ||  . | . | .  ||  . | . | . 
4   . | . | .  ||  . | . | .  ||  . | . | . 
5   . | . | .  ||  . | . | .  ||  . | . | . 
6   . | . | .  ||  . | . | .  ||  . | . | . 
7   . | . | .  ||  . | . | .  ||  . | . | . 
8   . | . | .  ||  . | . | .  ||  . | . | . 
9   . | . | .  ||  . | . | .  ||  . | . | . 

    1  2  3     4  5  6     7  8  9
  ▶ Tour de X | Zone : libre | Coups : 0


🤖 MonIA réfléchit...
  [IA] profondeur atteinte=8 | temps=7.88s

┌─────────────────────────────────────┐
│  MON COUP  →  colonne 5, ligne 5         │
│  (temps : 7.88s)                      │
└─────────────────────────────────────┘
  👉 Dictez ce coup à votre adversaire.

    1  2  3     4  5  6     7  8  9

1   . | . | .  ||  . | . | .  ||  . | . | . 
2   . | . | .  ||  . | . | .  ||  . | . | . 
3   . | . | .  ||  . | . | .  ||  . | . | . 
4  

## 5. 🤖 Test interne IA vs IA
Pour tester et calibrer votre IA avant le combat officiel.

In [ ]:
# ============================================================
# IA vs IA — Test et calibrage
# ============================================================

DEPTH_IA1   = 4      # profondeur IA-1 (votre IA)
DEPTH_IA2   = 3      # profondeur IA-2 (adversaire simulé)
SHOW_BOARD  = True   # False = plus rapide
USE_TIMED_TEST = True
TIME_TEST   = 5.0

def jouer_partie(joueur_x_timed, joueur_x_depth,
                 joueur_o_timed, joueur_o_depth,
                 nom_x='IA-X', nom_o='IA-O', show=True):
    board = Board()
    last_move = None
    timings = {PLAYER_X: [], PLAYER_O: []}
    if show: display_board(board)
    while not board.is_terminal():
        p = board.current_player
        t0 = time.time()
        if p == PLAYER_X:
            move = get_best_move_timed(board, time_limit=TIME_TEST) if joueur_x_timed \
                   else get_best_move(board, depth=joueur_x_depth)
        else:
            move = get_best_move_timed(board, time_limit=TIME_TEST) if joueur_o_timed \
                   else get_best_move(board, depth=joueur_o_depth)
        dt = time.time()-t0
        timings[p].append(dt)
        board = apply_move(board, move)
        last_move = move
        if show: display_board(board, last_move)
    return board, timings

print('── Partie 1 : IA-1 (X) vs IA-2 (O) ──')
b1, t1 = jouer_partie(USE_TIMED_TEST, DEPTH_IA1, False, DEPTH_IA2,
                       nom_x=NOM_IA, nom_o='IA-2', show=SHOW_BOARD)
r1 = b1.global_winner

print('── Partie 2 : IA-2 (X) vs IA-1 (O) ──')
b2, t2 = jouer_partie(False, DEPTH_IA2, USE_TIMED_TEST, DEPTH_IA1,
                       nom_x='IA-2', nom_o=NOM_IA, show=SHOW_BOARD)
r2 = b2.global_winner

# Score DVO
moves1, moves2 = b1.move_count, b2.move_count
moy = (moves1+moves2)/2

def pts(result, side, moves, moy):
    rapid = moves <= moy
    if result==side:    return 4 if rapid else 3
    elif result==DRAW:  return 1 if rapid else 0
    else:               return 1 if rapid else 0

s1 = pts(r1, PLAYER_X, moves1, moy) + pts(r2, PLAYER_O, moves2, moy)
s2 = pts(r1, PLAYER_O, moves1, moy) + pts(r2, PLAYER_X, moves2, moy)

print('\n' + '='*50)
sym = {PLAYER_X:'X', PLAYER_O:'O', DRAW:'='}
print(f'  Partie 1 : résultat={sym[r1]} | {moves1} coups')
print(f'  Partie 2 : résultat={sym[r2]} | {moves2} coups')
print(f'  Score DVO  {NOM_IA} : {s1} pts | IA-2 : {s2} pts')
if t1[PLAYER_X]: print(f'  Temps moy {NOM_IA} (X) : {sum(t1[PLAYER_X])/len(t1[PLAYER_X]):.2f}s')
if t2[PLAYER_O]: print(f'  Temps moy {NOM_IA} (O) : {sum(t2[PLAYER_O])/len(t2[PLAYER_O]):.2f}s')
print('='*50)

## 6. 🧑 Humain vs IA
Pour s'entraîner contre votre propre IA.

In [ ]:
# ============================================================
# HUMAIN vs IA
# ============================================================

HUMAIN_JOUE_X = True   # True = humain commence (X) | False = IA commence

board     = Board()
last_move = None
HUMAIN    = PLAYER_X if HUMAIN_JOUE_X else PLAYER_O

print('=== Humain vs IA ===')
print(f'Vous jouez : {"X" if HUMAIN==PLAYER_X else "O"}')
display_board(board)

while not board.is_terminal():
    if board.current_player == HUMAIN:
        ag = board.active_grid
        if ag:
            print(f'Vous devez jouer dans la sous-grille ({ag[0]+1},{ag[1]+1}).')
        else:
            print('Vous pouvez jouer partout (sous-grille libre).')
        move = None
        while move is None:
            c = input('  Votre colonne (1-9) : ').strip()
            l = input('  Votre ligne   (1-9) : ').strip()
            move = parse_move(c, l, board)
            if move is None: print('  Coup invalide, réessayez.')
    else:
        print('\n🤖 IA réfléchit...')
        t0   = time.time()
        move = mon_ia_joue(board)
        dt   = time.time()-t0
        gr,gc,lr,lc = move
        print(f'  IA joue colonne={gc*3+lc+1}, ligne={gr*3+lr+1} ({dt:.2f}s)')

    board     = apply_move(board, move)
    last_move = move
    display_board(board, last_move=last_move)

w = board.global_winner
if   w == HUMAIN: print('🏆 Vous gagnez !')
elif w == DRAW:   print('🤝 Match nul.')
else:             print('🤖 L\'IA gagne.')